# Personalized Learning Path Recommender
## Strategy: TF-IDF + Cosine Similarity (Optimized)

**Improvements over baseline (63.99):**
1. Course name repeated 3x in train features — boosts course-topic signal
2. Trigrams `(1,3)` instead of bigrams — captures longer review phrases
3. `max_features=60K` (was 30K) — richer vocabulary
4. `min_df=1` — keeps all course-specific technical terms
5. `float32` — faster CPU computation

In [ ]:
import pandas as pd
import numpy as np
import re
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

In [ ]:
# ── Config ──
DATA_DIR    = r"c:\Users\rithu\OneDrive\Desktop\HCL AMPlified Challenge\c215051c-6-Archive 4"
OUTPUT_PATH = r"c:\Users\rithu\OneDrive\Desktop\HCL AMPlified Challenge\submission.csv"

BATCH_SIZE = 500
TOP_K      = 10

In [ ]:
# ── Load data ──
print("Loading data...")
train  = pd.read_csv(f"{DATA_DIR}/train.csv")
test   = pd.read_csv(f"{DATA_DIR}/test.csv")
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
print(f"  Train: {train.shape},  Test: {test.shape}")
print(f"  Unique courses: {train['Course'].nunique()}")
train.head(3)

In [ ]:
# ── Text cleaning ──
def clean(text: str) -> str:
    """Lowercase, remove punctuation/digits, collapse whitespace."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
# ── Feature construction ──
# Repeat course name 3x so course-specific terms dominate TF-IDF weights
print("Building features...")
train["feat"] = (train["Course"].apply(clean) + " ") * 3 + train["Reviews"].apply(clean)
test["feat"]  = test["Reviews"].apply(clean)
print(f"  Sample train feat: {train['feat'].iloc[0][:120]}")

In [ ]:
# ── TF-IDF Vectorization ──
print("Fitting TF-IDF...")
t0 = time.time()

vec = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),      # trigrams for richer phrase coverage
    max_features=60_000,     # larger vocab than baseline 30K
    sublinear_tf=True,
    min_df=1,                # keep rare course-specific technical terms
    max_df=0.95,
    strip_accents="unicode",
    dtype=np.float32,
)

train_tfidf = normalize(vec.fit_transform(train["feat"]), norm="l2")
test_tfidf  = normalize(vec.transform(test["feat"]),      norm="l2")
print(f"  train: {train_tfidf.shape}, test: {test_tfidf.shape}  [{time.time()-t0:.1f}s]")

In [ ]:
# ── Top-10 recommendations via batch cosine similarity ──
print("Computing top-10 recommendations (batch mode)...")
t0 = time.time()

train_idx = train["Index"].values
results   = []
n_test    = test_tfidf.shape[0]

for start in range(0, n_test, BATCH_SIZE):
    end   = min(start + BATCH_SIZE, n_test)
    batch = test_tfidf[start:end]

    sim = (batch @ train_tfidf.T).toarray()

    for i, row in enumerate(sim):
        top = np.argpartition(row, -TOP_K)[-TOP_K:]
        top = top[np.argsort(row[top])[::-1]]
        results.append((test["Index"].iloc[start + i], train_idx[top].tolist()))

    if (start // BATCH_SIZE) % 5 == 0:
        pct     = end / n_test * 100
        elapsed = time.time() - t0
        eta     = elapsed / end * (n_test - end) if end > 0 else 0
        print(f"  [{pct:5.1f}%] {end}/{n_test}  elapsed={elapsed:.0f}s  ETA={eta:.0f}s")

print(f"  Done. Total: {time.time()-t0:.1f}s")

In [ ]:
# ── Build submission dataframe ──
submission = pd.DataFrame(results, columns=["Index", "Index_list"])
submission["Index_list"] = submission["Index_list"].apply(str)
print(f"Submission shape: {submission.shape}")
submission.head(3)

In [ ]:
# ── Validation checks ──
assert submission.shape[0] == test.shape[0],                        "Row count mismatch!"
assert list(submission.columns) == ["Index", "Index_list"],         "Column mismatch!"
assert (submission["Index"].values == test["Index"].values).all(),  "Index mismatch!"

lengths = submission["Index_list"].apply(lambda x: len(eval(x)))
assert (lengths == 10).all(), "Some rows don't have exactly 10 recommendations!"

print(f"All checks passed!")
print(f"Lengths — min: {lengths.min()}, max: {lengths.max()}")

In [ ]:
# ── Save submission ──
submission.to_csv(OUTPUT_PATH, index=False)
print(f"Saved → {OUTPUT_PATH}")

In [ ]:
# ── Preview vs sample_submission.csv ──
print("Our predictions:")
s_idx = sample["Index"].tolist()
display(submission[submission["Index"].isin(s_idx)].reset_index(drop=True))

print("\nSample submission (reference):")
display(sample)